# Mamba LLM
## What is Mamba LLM?
Mamba LLM is an architecture of neural networks designed for large language models. This a State Space Model (SSM) architecture, which is an alternative to the Transformer architecture used in models like GPT-3. Mamba LLM is designed to be more efficient and scalable, allowing for training on larger datasets and with fewer computational resources compared to traditional Transformer-based models.

### What are State Space Models (SSMs)?
State Space Models are recurrent models that maintain a hidden state that evolves over time. They process information selectively depending on the input, which enables them to filter out irrelevant information and focus on important features.

State equation:
$$h_t = A h_{t-1} + B x_t$$
Output equation:
$$y_t = C h_t + D x_t$$

A, B, C, D are learnable parameters that define how the hidden state and output are computed from the previous hidden state and current input.


## How it improves over transformer / current architecture?

Mamba LLM's architecture allows it to process sequences more efficiently than Transformers, which rely on self-attention mechanisms that can be computationally expensive, especially for long sequences. Transformers' self-attention mechanism has a quadratic complexity with respect to the sequence length, as each token attends to every other token in the sequence (time complexity is O(n^2)). In contrast, Mamba LLM's state space model uses a fixed-size hidden state that evolves over time, allowing it to process sequences in linear time with respect to the sequence length. This makes Mamba LLM more scalable and efficient for training on large datasets and handling long sequences (time complexity is O(n)).

## Open Source Libraries
[`mamba-ssm`](https://github.com/state-spaces/mamba) - official CUDA implementation.\
[`mambapy`](https://github.com/alxndrTL/mamba.py) - pure PyTorch implementation, suitable for CPU.\
Hugging Face Transformers - Mamba LLM is available (Mamba, MambaForCausalLM, Mamba2, FalconMamba)
`kotomamba

## Tutorials
- [DataCamp: An Introduction to the Mamba LLM Architecture: A New Paradigm in Machine Learning](https://www.datacamp.com/tutorial/introduction-to-the-mamba-llm-architecture)
- [A Visual Guide to Mamba and State Space Models](https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-mamba-and-state)
- [Hugging Face: Mamba LLM](https://huggingface.co/docs/transformers/v5.3.0/en/model_doc/mamba#mamba)

## Applications
- Edge AI
- Genomics
- Audio, speech, video processing
- Finance, time series analysis
- Code generation

## Demo
### mambapy example
Simple demonstration on how to use mambapy to **create** a Mamba LLM model and perform a forward pass with dummy made-up vocab and input data.


In [ ]:
!pip install mambapy
import torch
import torch.nn as nn
from mambapy.mamba import Mamba, MambaConfig


# ── 1. Data & Tokenizer ──────────────────────────────────


text = """
Mamba is a new LLM architecture that integrates the Structured State Space sequence (S4) model to manage lengthy data sequences. Combining the best features of recurrent, convolutional, and continuous-time models, S4 can effectively and efficiently simulate long-term dependencies. This allows it to handle irregularly sampled data, have unbounded context, and maintain computational efficiency throughout training and testing.

Expanding upon the S4 paradigm, Mamba brings about several noteworthy improvements, especially in handling time-variant operations. Its architecture revolves around a special selection mechanism that modifies the structured state space model (SSM) parameters according to the input.

As a result, Mamba may successfully filter out less important data by focusing only on crucial information within sequences. According to Wikipedia, "The model transitions from a time-invariant to a time-varying framework, which impacts both the computation and efficiency of the system."
"""

# Build vocabulary: every unique word gets an integer ID
words  = text.split()
vocab  = sorted(set(words))
w2i    = {w: i for i, w in enumerate(vocab)}
i2w    = {i: w for w, i in w2i.items()}
vocab_size = len(vocab)

print(f"Vocabulary ({vocab_size} words): {vocab}")

tokens = [w2i[w] for w in words] # every word into its ID
print(f"Tokens: {tokens[:10]}...")


def get_batch(batch_size=8, seq_len=6):
    batch_x, batch_y = [], []
    for _ in range(batch_size):
        start = torch.randint(0, len(tokens) - seq_len, (1,)).item()
        chunk = tokens[start : start + seq_len + 1]
        batch_x.append(chunk[:-1])   # all but last
        batch_y.append(chunk[1:])    # all but first  ← shifted by 1
    return (torch.tensor(batch_x),   # shape [B, seq_len]
            torch.tensor(batch_y))   # shape [B, seq_len]


# ── 2. Model ─────────────────────────────────────────────


d_model    = 32
config     = MambaConfig(d_model=d_model, n_layers=2)

class MambaLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding   = nn.Embedding(vocab_size, d_model)
        self.mamba       = Mamba(config)
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x shape:  [batch, seq_len]          (token IDs)
        x = self.embedding(x)    # → [batch, seq_len, d_model]
        x = self.mamba(x)        # → [batch, seq_len, d_model]
        x = self.output_head(x)  # → [batch, seq_len, vocab_size]
        return x

model = MambaLM()
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")


# ── 3. Train ─────────────────────────────────────────────

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

print("\nTraining...")
for step in range(500):
    x, target = get_batch()

    logits = model(x)
    # logits shape: [batch, seq_len, vocab_size]
    # CrossEntropyLoss wants: [batch, vocab_size, seq_len]
    loss = loss_fn(logits.transpose(1, 2), target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"  step {step:3d}  loss: {loss.item():.4f}")


# ── 4. Inference ─────────────────────────────────────────

def generate(start_word, n_words=6):
    model.eval()
    with torch.no_grad():
        result = [start_word]
        # Start with just the seed word as a 1-token sequence
        context = torch.tensor([[w2i[start_word]]])  # [1, 1]

        for _ in range(n_words):
            logits = model(context)          # [1, seq_len, vocab_size]
            last_logits = logits[0, -1, :]   # scores for the LAST position
            next_token = last_logits.argmax().item()  # pick highest score
            result.append(i2w[next_token])
            # Append predicted token to context for next step
            context = torch.cat([
                context,
                torch.tensor([[next_token]])
            ], dim=1)

    return " ".join(result)


print("\nGeneration examples:")
for seed in ["Mamba", "state", "model"]:
    print(f"  '{seed}' → {generate(seed)}")

Vocabulary (107 words): ['"The', '(S4)', '(SSM)', 'According', 'As', 'Combining', 'Expanding', 'Its', 'LLM', 'Mamba', 'S4', 'Space', 'State', 'Structured', 'This', 'Wikipedia,', 'a', 'about', 'according', 'allows', 'and', 'architecture', 'around', 'best', 'both', 'brings', 'by', 'can', 'computation', 'computational', 'context,', 'continuous-time', 'convolutional,', 'crucial', 'data', 'data,', 'dependencies.', 'effectively', 'efficiency', 'efficiently', 'especially', 'features', 'filter', 'focusing', 'framework,', 'from', 'handle', 'handling', 'have', 'impacts', 'important', 'improvements,', 'in', 'information', 'input.', 'integrates', 'irregularly', 'is', 'it', 'lengthy', 'less', 'long-term', 'maintain', 'manage', 'may', 'mechanism', 'model', 'models,', 'modifies', 'new', 'noteworthy', 'of', 'on', 'only', 'operations.', 'out', 'paradigm,', 'parameters', 'recurrent,', 'result,', 'revolves', 'sampled', 'selection', 'sequence', 'sequences.', 'several', 'simulate', 'space', 'special', 'sta


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



Model parameters: 26,859


### Hugging Face Transformers example

In [ ]:
import torch
from transformers import pipeline

pipeline = pipeline(
    task="text-generation",
    model="state-spaces/mamba-130m-hf",
    dtype=torch.float16,
    device=0
)
print('-' * 50)
pipeline("Plants create energy through a process known as")

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

MambaForCausalLM LOAD REPORT from: state-spaces/mamba-130m-hf
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------


[{'generated_text': 'Plants create energy through a process known asиз Toyota resulting invent\n                                 Toyota CASE Nelson relying theological lipids Toyota Nelson econom Try championship econom photosensitive econom econom awaited econom econom econom econom bathroom econom econom econom econom econom econom econom antiviral econom econom econom econom econom Guidelines econom bathroom Weight econom econom econom econom econom econom econom econom econom econom econom economCountisations Guidelines economayan Guidelines数 Toyota photosensitivefootball economattack Guidelines influx Guidelines Guidelines Guidelinesattack econom Guidelines Guidelines Guidelines econom Guidelines Guidelines Guidelines econom genetically Guidelines Guild Nelson econom photosensitiveSeason bathroom econom econom econom econom Weight econom econom econom econom econom econom Literature econom econom econom econom econom econom econom econom Guidelines powers GuidelinesCount Weight ec

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
model = AutoModelForCausalLM.from_pretrained("state-spaces/mamba-130m-hf", dtype=torch.float16, device_map="auto",)
input_ids = tokenizer("Plants create energy through a process known as", return_tensors="pt").to(model.device)

output = model.generate(**input_ids)
print('-' * 50)
print(tokenizer.decode(output[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

MambaForCausalLM LOAD REPORT from: state-spaces/mamba-130m-hf
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--------------------------------------------------
Plants create energy through a process known as Elliott Wilder solic configVIS 713 tagging inconsist config ----,440 identifiers fib lestderedderedderedelectrondereditian
